# Spam Detection — Thu thập & Tiền xử lý dữ liệu
**Thành viên:** B — Kỹ thuật dữ liệu  
**Tuần 1:** Tải dataset, làm sạch dữ liệu, TF-IDF vectorization  
**Dataset:** SMS Spam Collection (Kaggle)

## 1. Import thư viện

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

## 2. Tải dataset

In [2]:
df = pd.read_csv("spam.csv", encoding="latin-1")
df = df[['v1', 'v2']]
df.columns = ['label', 'text']
print(f"Shape ban đầu: {df.shape}")
df.head()

Shape ban đầu: (5572, 2)


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 3. Kiểm tra null và duplicate

In [3]:
print("=== Null values ===")
print(df.isnull().sum())
print(f"\n=== Duplicate rows: {df.duplicated().sum()} ===")

=== Null values ===
label    0
text     0
dtype: int64

=== Duplicate rows: 403 ===


## 4. Làm sạch dữ liệu

In [4]:
# Xóa duplicate
df = df.drop_duplicates()
print(f"Sau khi xóa duplicate: {df.shape[0]} dòng")

# Map nhãn sang 0/1
df["label"] = df["label"].map({"ham": 0, "spam": 1})

# Hàm làm sạch text: lowercase + xóa ký tự đặc biệt
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

# Xử lý các dòng clean_text rỗng hoặc null sau khi làm sạch
null_count  = df["clean_text"].isnull().sum()
empty_count = (df["clean_text"].str.strip() == "").sum()
print(f"Null trong clean_text: {null_count}")
print(f"Rỗng trong clean_text: {empty_count}")

df = df[df["clean_text"].notna() & (df["clean_text"].str.strip() != "")]
print(f"Sau khi xử lý null/rỗng: {df.shape[0]} dòng")
df.head()

Sau khi xóa duplicate: 5169 dòng
Null trong clean_text: 0
Rỗng trong clean_text: 3
Sau khi xử lý null/rỗng: 5166 dòng


,label,text,clean_text
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final...
3,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


## 5. TF-IDF Vectorization

In [5]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df["clean_text"])
y = df["label"]

print(f"Shape ma trận TF-IDF: {X.shape}")
print(f"Số nhãn y: {len(y)}")

Shape ma trận TF-IDF: (5166, 5000)
Số nhãn y: 5166


## 6. Xuất file CSV sạch

In [6]:
df.to_csv("clean_spam_dataset.csv", index=False)
print("Đã lưu: clean_spam_dataset.csv")
print(f"Shape: {df.shape}")
print(f"\nPhân phối nhãn:")
print(df['label'].value_counts())

Đã lưu: clean_spam_dataset.csv
Shape: (5166, 3)

Phân phối nhãn:
label
0    4513
1     653
Name: count, dtype: int64
